# Download và chuẩn hóa VizWiz-VQA trên Kaggle

Notebook chỉ tải ảnh và annotation của hai split **validation** và **test** từ nguồn chính thức, sau đó chuyển đổi sang định dạng `generic_vqa` của SelTDA. Cần bật **Internet** trong Kaggle.

Annotation test được lấy từ bộ đáp án tự đánh giá mà VizWiz công bố tháng 4/2026. Notebook không tải, không tạo và không đóng gói dữ liệu train.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/fantastichaha11/SelTDA.git'
BRANCH = 'feat/pseudo-label-filter'
REPO_DIR = Path('/kaggle/working/SelTDA')
OUTPUT_ROOT = Path('/kaggle/working/vizwiz')

INCLUDE_UNANSWERABLE = True
FORCE_DOWNLOAD = False
REMOVE_OLD_TRAIN_ARTIFACTS = True
DELETE_DOWNLOAD_ARCHIVES = False
CREATE_ARCHIVE = False
ARCHIVE_PATH = Path('/kaggle/working/vizwiz_seltda.zip')

VAL_IMAGES_URL = 'https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip'
TEST_IMAGES_URL = 'https://vizwiz.cs.colorado.edu/VizWiz_final/images/test.zip'
VAL_ANNOTATIONS_URL = 'https://vizwiz.cs.colorado.edu/VizWiz_final/vqa_data/Annotations/val.json'
TEST_ANNOTATIONS_URL = 'https://vizwiz.cs.colorado.edu/VizWiz_all_answers/VQA_test.json'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Output root: {OUTPUT_ROOT}')

In [ ]:
import subprocess
import sys

if not (REPO_DIR / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f'Using existing repository: {REPO_DIR}')

required_scripts = [REPO_DIR / 'scripts/download_vizwiz.py']
missing_scripts = [str(path) for path in required_scripts if not path.is_file()]
if missing_scripts:
    raise FileNotFoundError(f'Missing repository scripts: {missing_scripts}')
print(f'Repository ready: {REPO_DIR}')

In [ ]:
import shutil

sys.path.insert(0, str(REPO_DIR))
from scripts.download_vizwiz import download_file, extract_zip_files

downloads_dir = OUTPUT_ROOT / 'downloads'
annotations_dir = OUTPUT_ROOT / 'annotations'
downloads_dir.mkdir(parents=True, exist_ok=True)
annotations_dir.mkdir(parents=True, exist_ok=True)

download_specs = [
    (VAL_IMAGES_URL, downloads_dir / 'vizwiz_val.zip'),
    (TEST_IMAGES_URL, downloads_dir / 'vizwiz_test.zip'),
    (VAL_ANNOTATIONS_URL, annotations_dir / 'val.json'),
    (TEST_ANNOTATIONS_URL, annotations_dir / 'test.json'),
]
for url, target in download_specs:
    download_file(url, target, force=FORCE_DOWNLOAD)

image_suffixes = {'.jpg', '.jpeg', '.png'}
extract_zip_files(
    downloads_dir / 'vizwiz_val.zip',
    OUTPUT_ROOT / 'images/val',
    suffixes=image_suffixes,
    force=FORCE_DOWNLOAD,
)
extract_zip_files(
    downloads_dir / 'vizwiz_test.zip',
    OUTPUT_ROOT / 'images/test',
    suffixes=image_suffixes,
    force=FORCE_DOWNLOAD,
)

if REMOVE_OLD_TRAIN_ARTIFACTS:
    old_train_artifacts = [
        OUTPUT_ROOT / 'images/train',
        OUTPUT_ROOT / 'annotations/train.json',
        OUTPUT_ROOT / 'train.json',
        OUTPUT_ROOT / 'downloads/vizwiz_train.zip',
    ]
    for path in old_train_artifacts:
        if path.is_dir():
            shutil.rmtree(path)
            print(f'Removed old train directory: {path}')
        elif path.is_file():
            path.unlink()
            print(f'Removed old train file: {path}')

print('VizWiz val/test download and extraction completed.')

In [ ]:
import json

from dataset_adapters.generic_vqa import build_answer_list, normalize_answer, write_json

SPLIT_OFFSETS = {'val': 1_000_000, 'test': 2_000_000}

def normalized_answers(raw):
    answers = raw.get('answers', [])
    if not isinstance(answers, list):
        raise ValueError(f'VizWiz answers must be a list: {raw}')
    values = []
    for answer in answers:
        value = answer.get('answer', '') if isinstance(answer, dict) else answer
        normalized = normalize_answer(value)
        if normalized:
            values.append(normalized)
    if not values:
        raise ValueError('Downloaded annotation has no answers. Check the released-label URL.')
    return values

def majority_answer(answers):
    counts = {}
    for answer in answers:
        counts[answer] = counts.get(answer, 0) + 1
    return sorted(counts.items(), key=lambda item: (-item[1], item[0]))[0][0]

def convert_split(split_name):
    raw_path = OUTPUT_ROOT / 'annotations' / f'{split_name}.json'
    with raw_path.open(encoding='utf-8') as file:
        raw_records = json.load(file)
    if not isinstance(raw_records, list) or not raw_records:
        raise ValueError(f'VizWiz {split_name} annotation must be a non-empty list.')

    converted = []
    metadata = {}
    for index, raw in enumerate(raw_records):
        image_name = raw.get('image')
        question = str(raw.get('question', '')).strip()
        if not image_name or not question:
            raise ValueError(f'Missing image or question in {split_name}: {raw}')
        image_path = OUTPUT_ROOT / 'images' / split_name / image_name
        if not image_path.is_file():
            raise FileNotFoundError(f'Missing VizWiz image: {image_path}')
        answers = normalized_answers(raw)
        if not INCLUDE_UNANSWERABLE and majority_answer(answers) == 'unanswerable':
            continue
        question_id = SPLIT_OFFSETS[split_name] + index
        converted.append({
            'dataset': 'vizwiz',
            'image': f'{split_name}/{image_name}',
            'question': question,
            'question_id': question_id,
            'answer': answers,
        })
        metadata[str(question_id)] = {
            'answerable': int(raw.get('answerable', 1)),
            'answer_type': str(raw.get('answer_type', 'unknown')),
        }
    if not converted:
        raise ValueError(f'VizWiz {split_name} is empty after conversion.')
    return converted, metadata

converted_by_split = {}
metadata_by_split = {}
for split_name in ('val', 'test'):
    records, metadata = convert_split(split_name)
    converted_by_split[split_name] = records
    metadata_by_split[split_name] = metadata
    write_json(OUTPUT_ROOT / f'{split_name}.json', records)
    write_json(OUTPUT_ROOT / f'vizwiz_{split_name}_metadata.json', metadata)

# Chỉ dùng đáp án validation làm candidate vocabulary; không rò rỉ nhãn test.
write_json(OUTPUT_ROOT / 'answer_list.json', build_answer_list(converted_by_split['val']))
print('VizWiz val/test conversion completed.')

In [ ]:
required_outputs = [
    OUTPUT_ROOT / 'val.json',
    OUTPUT_ROOT / 'test.json',
    OUTPUT_ROOT / 'answer_list.json',
    OUTPUT_ROOT / 'vizwiz_val_metadata.json',
    OUTPUT_ROOT / 'vizwiz_test_metadata.json',
    OUTPUT_ROOT / 'images/val',
    OUTPUT_ROOT / 'images/test',
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(f'Missing converted VizWiz outputs: {missing_outputs}')

def load_json(path):
    with path.open(encoding='utf-8') as file:
        return json.load(file)

val_records = load_json(OUTPUT_ROOT / 'val.json')
test_records = load_json(OUTPUT_ROOT / 'test.json')
answer_list = load_json(OUTPUT_ROOT / 'answer_list.json')
val_metadata = load_json(OUTPUT_ROOT / 'vizwiz_val_metadata.json')
test_metadata = load_json(OUTPUT_ROOT / 'vizwiz_test_metadata.json')

if not val_records or not test_records or not answer_list:
    raise ValueError('Converted val, test, or answer list is empty.')

all_question_ids = [row['question_id'] for row in val_records + test_records]
if len(all_question_ids) != len(set(all_question_ids)):
    raise ValueError('question_id values are not unique across val and test.')

for split_name, records in [('val', val_records), ('test', test_records)]:
    for row in records:
        required_fields = {'image', 'question', 'question_id', 'answer'}
        if not required_fields <= row.keys():
            raise ValueError(f'Malformed {split_name} record: {row}')
        if not row['answer']:
            raise ValueError(f'Empty answers in {split_name}: {row}')
        image_path = OUTPUT_ROOT / 'images' / row['image']
        if not image_path.is_file():
            raise FileNotFoundError(f'Missing image: {image_path}')

for split_name, records, metadata in [
    ('val', val_records, val_metadata),
    ('test', test_records, test_metadata),
]:
    missing_metadata = [
        row['question_id'] for row in records if str(row['question_id']) not in metadata
    ]
    if missing_metadata:
        raise ValueError(f'Missing {split_name} metadata IDs: {missing_metadata[:10]}')

print(f'Validation records: {len(val_records):,}')
print(f'Test records: {len(test_records):,}')
print(f'Candidate answers: {len(answer_list):,}')
print(f'Validation metadata rows: {len(val_metadata):,}')
print(f'Test metadata rows: {len(test_metadata):,}')
if (OUTPUT_ROOT / 'images/train').exists():
    raise RuntimeError('Unexpected train image directory remains in OUTPUT_ROOT.')

In [ ]:
from IPython.display import display
from PIL import Image

sample = test_records[0]
sample_image_path = OUTPUT_ROOT / 'images' / sample['image']
print(json.dumps(sample, indent=2, ensure_ascii=False))
display(Image.open(sample_image_path).convert('RGB'))

In [ ]:
import shutil
from zipfile import ZIP_DEFLATED, ZipFile

if DELETE_DOWNLOAD_ARCHIVES:
    downloads_dir = OUTPUT_ROOT / 'downloads'
    if downloads_dir.is_dir():
        shutil.rmtree(downloads_dir)
        print(f'Removed downloaded ZIP files: {downloads_dir}')

if CREATE_ARCHIVE:
    files_to_archive = [
        OUTPUT_ROOT / 'val.json',
        OUTPUT_ROOT / 'test.json',
        OUTPUT_ROOT / 'answer_list.json',
        OUTPUT_ROOT / 'vizwiz_val_metadata.json',
        OUTPUT_ROOT / 'vizwiz_test_metadata.json',
    ]
    for split_name in ('val', 'test'):
        split_dir = OUTPUT_ROOT / 'images' / split_name
        if split_dir.is_dir():
            files_to_archive.extend(path for path in split_dir.rglob('*') if path.is_file())
    with ZipFile(ARCHIVE_PATH, 'w', compression=ZIP_DEFLATED, allowZip64=True) as archive:
        for path in files_to_archive:
            archive.write(path, path.relative_to(OUTPUT_ROOT))
    print(f'Archive created: {ARCHIVE_PATH}')
else:
    print(f'Dataset is ready at: {OUTPUT_ROOT}')
    print('Set CREATE_ARCHIVE=True only if a ZIP artifact is needed; VizWiz images are large.')